In [ ]:
pip install tensorflow category_encoders

In [9]:
import tensorflow as tf
import category_encoders as ce
import numpy as np
import pandas as pd
from tensorflow import keras
from tensorflow.keras.layers import Input, Embedding, Dense, Concatenate
from tensorflow.keras.optimizers.schedules import PolynomialDecay
import copy
from numpy import unique
from sklearn.preprocessing import LabelEncoder
from keras.models import Model
from keras.layers import Input
from keras.layers import Dense
from keras.layers import Embedding
from keras.layers import concatenate
from sklearn.metrics import mean_squared_error
from keras.utils import plot_model
import math
from tensorflow.keras.layers import Reshape
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import BatchNormalization
from tensorflow.keras.layers import Conv2D
from tensorflow.keras.layers import MaxPooling2D
from tensorflow.keras.layers import Activation
from tensorflow.keras.layers import Dropout
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.layers import LSTM
import matplotlib.pyplot as plt
#from keras.utils import plot_model
import os
from sklearn.preprocessing import RobustScaler
from collections import Counter
import tensorflow as tf
from tensorflow.keras import optimizers
from tensorflow.keras import metrics
from tensorflow.keras import losses
from transformers import pipeline
from keras.layers import Softmax
from tensorflow.keras.utils import plot_model
from tensorflow.keras import regularizers
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import pickle

In [10]:
def scale_data(df):
    numerical_cols = ["LOTS_NUMBER", "CRIT_PRICE_WEIGHT", "NUMBER_OFFERS", "NUMBER_TENDERS_SME"]
    scaler = RobustScaler()

    numeric_data = df[numerical_cols].values.reshape((len(df), len(numerical_cols)))
    scaled_num_data = scaler.fit_transform(numeric_data)
    df_numeric = pd.DataFrame(data=scaled_num_data,columns = numerical_cols)
    df[numerical_cols] = df_numeric

    return df

In [11]:
def train_validate_test_split(df, train, validate):
    target_col = "AWARD_VALUE_EURO_FIN_1"
    #df = df.sort_values(by = ["date_conclusion_contract"], axis = 0, ascending = True)
    train_indice = int(train * len(df))
    validate_indice = int(validate * len(df)) + train_indice
    print(train_indice, validate_indice)

    train_set = df[:train_indice]
    val_set = df[train_indice:validate_indice]
    test_set = df[validate_indice:]

    X_train = train_set.drop(columns = [target_col]).values
    y_train = train_set[target_col].values

    X_val = val_set.drop(columns = [target_col]).values
    y_val = val_set[target_col].values

    X_test = test_set.drop(columns = [target_col]).values
    y_test = test_set[target_col].values

    return X_train, y_train, X_val, y_val, X_test, y_test

In [12]:
def encode_data(df, target_column="award_contract/val_total"):

    base_n_encoder_cols = ["ISO_COUNTRY_CODE", "MAIN_ACTIVITY", "CPV"]
    one_hot_cols = ["CAE_TYPE", "B_ON_BEHALF", "B_AWARDED_BY_CENTRAL_BODY", "TYPE_OF_CONTRACT",
                    "B_FRA_AGREEMENT", "B_EU_FUNDS", "TOP_TYPE", "CRIT_CODE", "lots"]

    encoder = ce.BaseNEncoder(cols=base_n_encoder_cols, return_df=True, base=2)
    df = encoder.fit_transform(df)
    df = pd.get_dummies(df, columns=(one_hot_cols), drop_first=True, dtype=int)

    return df

In [13]:
def filter_outliers(df, upper, lower, target_col = "AWARD_VALUE_EURO_FIN_1"):
    """This function only works on numerical columns"""
    data_array = np.sort(df[target_col].to_numpy())
    cut_off_low_indice = math.floor(lower * len(data_array))
    cut_off_high_indice = math.floor(upper * len(data_array)) - 1
    low_amount = data_array[cut_off_low_indice]
    high_amount = data_array[cut_off_high_indice]

    outlier_indices = []

    for i in range(0, len(df)):
        if df[target_col].iloc[i] > high_amount:
            outlier_indices.append(i)
        elif df[target_col].iloc[i] < low_amount:
            outlier_indices.append(i)
        else:
            continue

    print("{} rows were dropped because of outliers, with high amount = {}, and low amount = {}".format(len(outlier_indices), high_amount, low_amount))
    df = df.drop(labels = outlier_indices, axis = 0)
    return df

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [15]:
#pandas settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [16]:
df = pd.read_pickle("/content/drive/MyDrive/Thesis/df_from_csv_preprocessed.pickle")

In [ ]:
df.head()

In [ ]:
def create_model_1(input_dimension, X_train, y_train, X_val, y_val, X_test, y_test, start_learning_rate, end_learning_rate, activation_function, loss_function, batch_size, epochs = 50):
    #let's play around a little more with the keras model
    metrics = ["mse", "mae", "R2Score"]

    #define the layers
    input_num_cat = Input(shape=input_dimension)
    x = Dense(32, activation = activation_function)(input_num_cat)
    x = Dropout(rate=0.2)(x)
    x = Dense(8, activation = activation_function)(x)
    x = Dropout(rate=0.2)(x)
    x = Dense(32, activation = activation_function)(x)
    regression_layer = Dense(1, activation="linear")(x)
    model_num_cat = Model(inputs = [input_num_cat],
                          outputs = regression_layer)

    num_train_steps = len(X_train) / batch_size * epochs
    lr_scheduler = PolynomialDecay(
        initial_learning_rate=start_learning_rate, end_learning_rate=end_learning_rate, decay_steps=num_train_steps
    )

    #checkpoint_path = "Results/Models/"
    #cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
    #                                                 verbose=1)

    optimizer = Adam(learning_rate=lr_scheduler)
    num_train_steps

    model_num_cat.compile(loss=loss_function,
                          optimizer = optimizer,
                          metrics = metrics)
    model_num_cat.summary()

    history = model_num_cat.fit(x = [X_train], y=y_train,
                                  validation_data = (X_val, y_val),
                                  epochs = epochs,
                                  batch_size = batch_size)

    y_pred = model_num_cat.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)

    metric = tf.keras.metrics.R2Score()
    metric.update_state(y_test.reshape(-1,1), y_pred)
    r2_result = metric.result().numpy()

    return model_num_cat, history, mae, r2_result, mse

In [ ]:
def create_model(model_number, input_dimension, X_train, y_train, X_val, y_val, X_test, y_test, start_learning_rate, end_learning_rate, activation_function, loss_function, batch_size, epochs = 50):
  #let's play around a little more with the keras model
  metrics = ["mse", "mae", "R2Score"]
  if model_number == 1:
    #define the layers
    input_num_cat = Input(shape=input_dimension)
    x = Dense(32, activation = activation_function)(input_num_cat)
    x = Dropout(rate=0.2)(x)
    x = Dense(8, activation = activation_function)(x)
    x = Dropout(rate=0.1)(x)
    x = Dense(32, activation = activation_function)(x)
    regression_layer = Dense(1, activation="linear")(x)
    model_num_cat = Model(inputs = [input_num_cat],
                          outputs = regression_layer)

  elif model_number == 2:
    #define the layers
    input_num_cat = Input(shape=input_dimension)
    x = Dense(32, activation = activation_function)(input_num_cat)
    x = Dropout(rate=0.2)(x)
    x = Dense(16, activation = activation_function)(x)
    x = Dropout(rate=0.2)(x)
    x = Dense(8, activation = activation_function)(x)
    x = Dropout(rate=0.1)(x)
    x = Dense(16, activation = activation_function)(x)
    x = Dropout(rate=0.1)(x)
    x = Dense(32, activation = activation_function)(x)
    regression_layer = Dense(1, activation="linear")(x)
    model_num_cat = Model(inputs = [input_num_cat],
                          outputs = regression_layer)

  elif model_number == 3:

    #let's play around a little more with the keras model
    metrics = ["mse", "mae", "R2Score"]

    #define the layers
    input_num_cat = Input(shape=input_dimension)
    x = Dense(226, activation = activation_function)(input_num_cat)
    x = Dropout(rate=0.2)(x)
    x = Dense(128, activation = activation_function)(x)
    x = Dropout(rate=0.2)(x)
    x = Dense(64, activation = activation_function)(x)
    x = Dropout(rate=0.2)(x)
    x = Dense(32, activation = activation_function)(x)
    x = Dropout(rate=0.2)(x)
    x = Dense(64, activation = activation_function)(x)
    x = Dropout(rate=0.2)(x)
    x = Dense(32, activation = activation_function)(x)
    regression_layer = Dense(1, activation="linear")(x)
    model_num_cat = Model(inputs = [input_num_cat],
                          outputs = regression_layer)

  num_train_steps = len(X_train) / batch_size * epochs
  lr_scheduler = PolynomialDecay(
      initial_learning_rate=start_learning_rate, end_learning_rate=end_learning_rate, decay_steps=num_train_steps
  )

  #checkpoint_path = "Results/Models/"
  #cp_callback = tf.keras.callbacks.ModelCheckpoint(filepath=checkpoint_path,
  #                                                 verbose=1)

  optimizer = Adam(learning_rate=lr_scheduler)
  num_train_steps

  model_num_cat.compile(loss=loss_function,
                        optimizer = optimizer,
                        metrics = metrics)
  model_num_cat.summary()

  history = model_num_cat.fit(x = [X_train], y=y_train,
                                validation_data = (X_val, y_val),
                                epochs = epochs,
                                batch_size = batch_size)

  y_pred = model_num_cat.predict(X_test)
  mae = mean_absolute_error(y_test, y_pred)
  mse = mean_squared_error(y_test, y_pred)

  metric = tf.keras.metrics.R2Score()
  metric.update_state(y_test.reshape(-1,1), y_pred)
  r2_result = metric.result().numpy()

  return model_num_cat, history, mae, r2_result, mse

In [ ]:
#create the dataset
df = pd.read_pickle("/content/drive/MyDrive/Thesis/final_data_set_from_csv.pickle")
pd.isna(df).sum();

In [ ]:
#df = df.drop(columns=["ID_LOT", "ID_NOTICE_CAN"])
crit_price_weight_empty = list(df.loc[pd.isna(df["CRIT_PRICE_WEIGHT"]) == True].index.values)
df = df.drop(labels=crit_price_weight_empty, axis = 0).reset_index(drop=True)
df = filter_outliers(df, upper = 0.90, lower=0.30)
#df = scale_data(df)
#ENCODE DATA
#df = encode_data(df)
len(df)

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = train_validate_test_split(df, 0.6, 0.2)
print(np.isnan(X_train).any(), np.isnan(y_train).any(),
      np.isnan(X_val).any(), np.isnan(y_val).any(),
      np.isnan(X_test).any(), np.isnan(y_test).any())

In [ ]:
activation_functions = ["selu", "relu"]
loss_functions = ["mae", "mse"]

#COMPILE AND TRAIN THE MODEL
results_architectures = {}
for i in range(1, 4):
  for activation_function in activation_functions:
    for loss_function in loss_functions:
      model, history, mae, r2_result, mse = create_model(i, input_dimension=X_train.shape[1],
             X_train = X_train,
             y_train = y_train,
             X_val = X_val,
             y_val = y_val,
             X_test = X_test,
             y_test = y_test,
             start_learning_rate= 0.01,
             end_learning_rate = 0.0001,
             epochs = 30,
             batch_size = 32,
             activation_function=activation_function,
             loss_function=loss_function)

      results_architectures[str(i) + "_" + activation_function + "_" + loss_function] = {
          "model": model,
          "history": history,
          "mae_test": mae,
          "mse_test": mse,
          "R2_test": r2_result}

In [ ]:
with open("/content/drive/MyDrive/Thesis/results_model_architectures.pickle", "wb") as f:
  pickle.dump(results_architectures, f)

In [ ]:
results_architectures

In [ ]:
#GET TEST SCORES PER MODEL
for key in results_architectures.keys():
  print("Model: {}".format(key), "\n",
        "mae test score: ", "{:e}".format(results_architectures[key]["mae_test"]), "\n",
        "mse test score: ", "{:e}".format(results_architectures[key]["mse_test"]), "\n",
        "r2 test score: ", results_architectures[key]["R2_test"], "\n")

In [ ]:
for key in results_architectures.keys():
  with open("/content/drive/MyDrive/Thesis/Kopie van {}.pickle".format(key), "rb") as f:
    print("model results for model {}: ".format(key))
    key = pickle.load(f)

    print("lowest model loss: ", np.min(key["history"].history["loss"]),
          "lowest model mse: ", np.min(key["history"].history["mse"]),
          "lowest model mae: ", np.min(key["history"].history["mae"]))
    print("\n")


In [ ]:
def plot_metrics(ledger, save_path = None):
  #model results is a dict from a keras object
  model_results = ledger.history
  plt.figure(figsize=(10, 10))

  # Create the top row of subplots
  ax1 = plt.subplot(3, 1, 1)
  ax2 = plt.subplot(3, 1, 2)
  ax3 = plt.subplot(3, 1, 3)
  axes = [ax1, ax2, ax3]
  for i, key in enumerate(list(model_results.keys())[:int(0.5*len(model_results.keys()))]):
    loss_train = model_results[key]
    loss_val   = model_results["val_" + key]
    epochs = range(0,len(loss_train))

    axes[i].plot(epochs, loss_train, "g", label = "Training {}".format(key))
    axes[i].plot(epochs, loss_val, "b", label = "Validation {}".format("val_" + key))
    axes[i].title.set_text("Training and validation {}".format(key))
    axes[i].set_xlabel("Epochs")
    axes[i].set_ylabel("{}".format(key))
    axes[i].legend()
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, format='png')

In [ ]:
def plot_metrics(ledger, save_path=None):
    model_results = ledger.history
    plt.figure(figsize=(10, 14))

    # Create the top row of subplots
    ax1 = plt.subplot(3, 1, 1)
    ax2 = plt.subplot(3, 1, 2)
    ax3 = plt.subplot(3, 1, 3)
    axes = [ax1, ax2, ax3]

    # Use the same line styles for all plots
    line_styles = ['-', '-.']
    markers = ['o', 's']

    for i, key in enumerate(list(model_results.keys())[:int(0.5 * len(model_results.keys()))]):
        loss_train = model_results[key]
        loss_val = model_results["val_" + key]
        epochs = range(0, len(loss_train))

        # Use different markers for training and validation
        axes[i].plot(epochs, loss_train, color='grey', linestyle=line_styles[0], label="Training {}".format(key), marker=markers[0])
        axes[i].plot(epochs, loss_val, color='grey', linestyle=line_styles[1], label="Validation {}".format("val_" + key), marker=markers[1])
        axes[i].title.set_text("Training and validation {}".format(key))
        axes[i].set_xlabel("Epochs")
        axes[i].set_ylabel("{}".format(key))
        axes[i].legend()

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, format='png')
    plt.show()

# Example usage:
# plot_metrics(your_ledger_object, save_path="your_plot.png")


In [ ]:
#MAKE SOME GRAPHS
keys = list(results_architectures.keys())
plot_metrics(results_architectures[keys[0]]["history"])

In [ ]:
#with open("/content/drive/MyDrive/Thesis/256-128-64-32-64-32_selu_mse.pickle", "wb") as f:
#  pickle.dump(results, f)

In [ ]:
import matplotlib.pyplot as plt

def plot_metrics(results_architectures, save_path=None):
    num_models = len(results_architectures)
    num_metrics = 2  # Number of metrics to plot (MAE and MSE)
    num_rows = num_models * num_metrics

    plt.figure(figsize=(15, 2.5 * num_models))

    for i, (model_name, model_results) in enumerate(results_architectures.items()):
        history = model_results['history'].history  # Access the 'history' attribute

        # Create subplots for MAE and MSE
        for j, metric in enumerate(['mae', 'mse']):
            ax = plt.subplot(num_rows, 2, i * 2 * num_metrics + j + 1)

            # Extract metric values for training and validation
            metric_train = history[metric]
            metric_val = history["val_" + metric]
            epochs = range(1, len(metric_train) + 1)

            # Use the same line styles for all plots
            line_styles = ['-', '-.']
            markers = ['o', 's']

            # Plot training and validation metrics
            ax.plot(epochs, metric_train, label="Training {}".format(metric), color="grey", linestyle=line_styles[0], marker=markers[0])
            ax.plot(epochs, metric_val, label="Validation {}".format("val_" + metric), color="grey", linestyle=line_styles[1], marker=markers[1])

            if model_name.split("_")[0] == "1":
              model_type = "A"
            elif model_name.split("_")[0] == "2":
              model_type = "B"
            else:
              model_type = "C"

            if model_name.split("_")[1] == "selu":
              activation = "SeLU"
            else:
              activation = "ReLU"

            if model_name.split("_")[2] == "mae":
              loss = "mae"
            else:
              loss = "mse"

            ax.set_title("Model {} - activation {}, loss {}".format(model_type, activation, loss))
            ax.set_xlabel("Epochs")
            ax.set_ylabel(metric.upper())
            ax.legend()

    # Adjust the vertical space between subplots
    plt.subplots_adjust(hspace=0.05)  # Adjust the value as needed

    #plt.tight_layout()

    if save_path:
        plt.savefig(save_path, format='png')

    plt.show()

# Example usage:
# Define your results_architectures dictionary
# results_architectures = {'model_1': {'history': ..., 'other_stuff': ...}, 'model_2': {'history': ..., 'other_stuff': ...}, ...}
# plot_metrics(results_architectures, save_path="your_plot.png")


In [ ]:
import matplotlib.pyplot as plt

def plot_metrics(results_architectures, save_path=None):
    num_models = len(results_architectures)
    num_metrics = 2  # Number of metrics to plot (MAE and MSE)
    num_rows = num_models * num_metrics

    plt.figure(figsize=(20, 2.5 * num_models))

    for i, (model_name, model_results) in enumerate(results_architectures.items()):
        history = model_results['history'].history  # Access the 'history' attribute

        # Create subplots for MAE and MSE
        for j, metric in enumerate(['mae', 'mse']):
            ax = plt.subplot(num_rows, 2, i * 2 * num_metrics + j + 1)

            # Extract metric values for training and validation
            metric_train = history[metric]
            metric_val = history["val_" + metric]
            epochs = range(1, len(metric_train) + 1)

            # Use the same line styles for all plots
            line_styles = ['-', '-.']
            markers = ['o', 's']

            # Plot training and validation metrics
            ax.plot(epochs, metric_train, label="Training {}".format(metric), color="grey", linestyle=line_styles[0], marker=markers[0])
            ax.plot(epochs, metric_val, label="Validation {}".format("val_" + metric), color="grey", linestyle=line_styles[1], marker=markers[1])

            if model_name.split("_")[0] == "1":
                model_type = "A"
            elif model_name.split("_")[0] == "2":
                model_type = "B"
            else:
                model_type = "C"

            if model_name.split("_")[1] == "selu":
                activation = "SeLU"
            else:
                activation = "ReLU"

            if model_name.split("_")[2] == "mae":
                loss = "mae"
            else:
                loss = "mse"

            ax.set_title("Model {} - activation {}, loss {}".format(model_type, activation, loss))
            ax.set_xlabel("Epochs")
            ax.set_ylabel(metric.upper())

    # Create a single legend outside the loop
    handles, labels = ax.get_legend_handles_labels()
    plt.figlegend(handles, labels, loc='upper right', bbox_to_anchor=(0.56, 0.91))

    # Adjust the vertical space between subplots
    plt.subplots_adjust(hspace=0.05)  # Adjust the value as needed

    #plt.tight_layout()

    if save_path:
        plt.savefig(save_path, format='png')

    plt.show()

# Example usage:
# Define your results_architectures dictionary
# results_architectures = {'model_1': {'history': ..., 'other_stuff': ...}, 'model_2': {'history': ..., 'other_stuff': ...}, ...}
# plot_metrics(results_architectures, save_path="your_plot.png")


In [ ]:
plot_metrics(results_architectures)

In [ ]:
#take variables out iteratively to get model results in steps
cpv_cols = [col for col in df.columns if "cpv_" in col]
country_cols = [col for col in df.columns if "country" in col]
type_contract_cols = [col for col in df.columns if "TYPE_OF_CONTRACT" in col]
ca_type_cols = [col for col in df.columns if "CAE_TYPE" in col]
procedure = [col for col in df.columns if "TOP_TYPE" in col]
n_offers = ["NUMBER_OFFERS"]
crit_code = ["CRIT_CODE_M"]


leave_one_out_cols = {"cpv_cols": cpv_cols,
                      "country_cols":country_cols,
                      "type_contract_cols":type_contract_cols,
                      "ca_type_cols": ca_type_cols,
                      "procedure": procedure,
                      "number_of_offers": n_offers,
                      "crit_code": crit_code
                      }

results = {}
for i, feature in enumerate(leave_one_out_cols.keys()):
    df_subset = copy.deepcopy(df.drop(columns = leave_one_out_cols[feature], axis=0))

    print(i, feature)

    X_train, y_train, X_val, y_val, X_test, y_test = train_validate_test_split(df_subset, 0.6, 0.2)
    input_dimension = X_train.shape[1]

    #COMPILE AND TRAIN THE MODEL
    model, history, mae, r2_result, mse = create_model(model_number = 1,input_dimension=X_train.shape[1],
             X_train = X_train,
             y_train = y_train,
             X_val = X_val,
             y_val = y_val,
             X_test = X_test,
             y_test = y_test,
             start_learning_rate= 0.005,
             end_learning_rate = 0.0001,
             epochs = 30,
             batch_size = 32,
             activation_function="selu",
             loss_function="mse")

    results["without {}".format(feature)] = {"model": model,
                                             "history": history,
                                             "mae_test": mae,
                                             "mse_test": mse,
                                             "R2_test": r2_result}

In [ ]:
#with open("/content/drive/MyDrive/Thesis/COOTV.pickle", "wb") as f:
#  pickle.dump(results, f)

In [ ]:
for key in results_architectures.keys():
  with open("/content/drive/MyDrive/Thesis/Kopie van {}.pickle".format(key), "rb") as f:
    print("model results for model {}: ".format(key))
    key = pickle.load(f)

    print("lowest model loss: ", np.min(key["history"].history["loss"]),
          "lowest model mse: ", np.min(key["history"].history["mse"]),
          "lowest model mae: ", np.min(key["history"].history["mae"]))
    print("\n")

In [ ]:
for key in results.keys():
  print("feature excluded: {}".format(key), "\n"
        "test mae: ", "{:e}".format(results[key]["mae_test"]), "\n"
        "test mse: ", "{:e}".format(results[key]["mse_test"]), "\n"
        "test R2: ", results[key]["R2_test"], "\n")

In [ ]:
results

In [ ]:
import tensorflow as tf
from tensorflow.keras import Input, Model
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import LearningRateScheduler
from sklearn.metrics import mean_absolute_error, mean_squared_error
from tensorflow.keras.optimizers.schedules import PolynomialDecay

def linear_decay(epoch, current_learning_rate, end_learning_rate, total_epochs):
    decay = (current_learning_rate - end_learning_rate) / total_epochs
    new_learning_rate = current_learning_rate - decay
    return max(new_learning_rate, end_learning_rate)

def create_model(input_dimension, X_train, y_train, X_val, y_val, X_test, y_test, start_learning_rate, end_learning_rate, activation_function, loss_function, batch_size, epochs=50):
    metrics = ["mse", "mae", "R2Score"]

    #define the layers
    input_num_cat = Input(shape=input_dimension)
    x = Dense(32, activation = activation_function)(input_num_cat)
    x = Dropout(rate=0.2)(x)
    x = Dense(16, activation = activation_function, kernel_regularizer=tf.keras.regularizers.L1L2(l1=1e-3, l2=1e-3))(x)
    x = Dropout(rate=0.2)(x)
    x = Dense(8, activation = activation_function, kernel_regularizer=tf.keras.regularizers.L1L2(l1=1e-3, l2=1e-3))(x)
    x = Dropout(rate=0.2)(x)
    x = Dense(16, activation = activation_function)(x)
    x = Dropout(rate=0.2)(x)
    x = Dense(32, activation = activation_function)(x)
    regression_layer = Dense(1, activation="linear")(x)
    model_num_cat = Model(inputs = [input_num_cat],
                          outputs = regression_layer)

    num_train_steps = len(X_train) / batch_size * epochs

    def lr_schedule(epoch, lr):
        return linear_decay(epoch, lr, end_learning_rate, epochs)

    lr_scheduler = LearningRateScheduler(lr_schedule)

    optimizer = Adam(learning_rate=start_learning_rate)
    model_num_cat.compile(loss=loss_function, optimizer=optimizer, metrics=metrics)

    model_num_cat.summary()

    history = model_num_cat.fit(x=[X_train], y=y_train, validation_data=(X_val, y_val), epochs=epochs, batch_size=batch_size, callbacks=[lr_scheduler])

    y_pred = model_num_cat.predict(X_test)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)

    metric = tf.keras.metrics.R2Score()
    metric.update_state(y_test.reshape(-1, 1), y_pred)
    r2_result = metric.result().numpy()

    return model_num_cat, history, mae, r2_result, mse

# Example usage:
# Adjust your hyperparameters accordingly
model, history, mae, r2, mse = create_model(input_dimension, X_train, y_train, X_val, y_val, X_test, y_test, start_learning_rate=0.03,
                                            end_learning_rate=0.00001, activation_function='relu', loss_function='mae', batch_size=64, epochs=50)

In [ ]:
results_finetuned = {"model": model,
                    "history": history,
                    "mae_test": mae,
                    "mse_test": mse,
                    "R2_test": r2_result}

In [ ]:
results_finetuned_model_A = copy.deepcopy(results_finetuned)
#with open("/content/drive/MyDrive/Thesis/results_finetuned_model_A.pickle", "wb") as f:
#  pickle.dump(results_finetuned_model_A, f)


In [ ]:
df = pd.read_pickle("/content/drive/MyDrive/Thesis/final_data_set_from_csv.pickle")
df.head()

In [ ]:
X_train, y_train, X_val, y_val, X_test, y_test = train_validate_test_split(df, 0.6, 0.2)

In [ ]:
X_train = torch.tensor(X_train, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32)
X_val = torch.tensor(X_val, dtype=torch.float32)
y_val = torch.tensor(y_val, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_test = torch.tensor(y_test, dtype=torch.float32)

In [ ]:
X_train.shape

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class PyTorchModel(nn.Module):
    def __init__(self, input_size, hidden_size1, hidden_size2):
        super(PyTorchModel, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size1)
        self.relu1 = nn.ReLU()
        self.dropout1 = nn.Dropout(p=0.2)
        self.fc2 = nn.Linear(hidden_size1, hidden_size2)
        self.relu2 = nn.ReLU()
        self.dropout2 = nn.Dropout(p=0.2)
        self.fc3 = nn.Linear(hidden_size2, hidden_size1)
        self.regression_layer = nn.Linear(hidden_size1, 1)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.dropout1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        x = self.dropout2(x)
        x = self.fc3(x)
        regression_output = self.regression_layer(x)
        return regression_output

# Instantiate your PyTorch model
input_size = X_train[0].shape[0] # Update with your actual input dimension
hidden_size1 = 32
hidden_size2 = 8

pytorch_model = PyTorchModel(input_size, hidden_size1, hidden_size2)

# You can print the model summary if needed
print(pytorch_model)

In [ ]:
import torch.optim as optim
epochs = 2
# Assuming X_train and y_train are PyTorch tensors
criterion = nn.MSELoss()
optimizer = optim.Adam(pytorch_model.parameters(), lr=0.001)

# Training loop
for epoch in range(epochs):
    optimizer.zero_grad()
    outputs = pytorch_model(X_train)
    loss = criterion(outputs, y_train)
    loss.backward()
    optimizer.step()

# After training, you can use the PyTorch model for predictions
with torch.no_grad():
    pytorch_model.eval()
    y_pred_pytorch = pytorch_model(X_test)